In [3]:
import pytorch_lightning as pl

MODEL_NAME = "beomi/KcELECTRA-base"

# 2.1. 공통 전처리 함수 및 라벨 정의
LABELS = ['불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경', '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적', '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움', '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음', '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움', '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰']

# 3.4. Pytorch Lightning 모델(BaseTagger) 정의
class BaseTagger(pl.LightningModule):
    def __init__(self, model_name=MODEL_NAME, lr=2e-5, weight_decay=0.01,
                 n_training_steps=None, n_warmup_steps=None, dropout_rate=0.1):
        super().__init__()
        self.save_hyperparameters()
        self.electra = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Dropout(p=self.hparams.dropout_rate),
            nn.Linear(self.electra.config.hidden_size, len(LABELS))
        )
        self.criterion = nn.BCELoss()

    def forward(self, input_ids, attention_mask, labels=None):
        output = self.electra(input_ids, attention_mask=attention_mask)
        logits = self.classifier(output.last_hidden_state[:, 0, :])
        probs = torch.sigmoid(logits)

        if labels is not None:
            loss = self.criterion(probs, labels)
            return loss, probs
        return None, probs

    def training_step(self, batch, batch_idx):
        loss, _ = self(**batch)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        loss, _ = self(**batch)
        self.log("val_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay
        )
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=self.hparams.n_warmup_steps,
            num_training_steps=self.hparams.n_training_steps
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "interval": "step"}}

In [4]:
import torch
import torch.nn as nn
import os
from transformers import AutoModel, AutoTokenizer

# BaseTagger 클래스 정의 필요

# ===================================================================
# 변환을 위한 설정
# ===================================================================
# 1. 불러올 체크포인트 파일 경로
CKPT_PATH = "./model/model_B/best_model_B_minmax_0.2.ckpt"

# 2. Hugging Face 형식으로 저장할 디렉토리 경로
SAVE_DIR = "./hf_kpoem"

# 3. 필요한 경우, 원본 모델의 하이퍼파라미터를 여기에 명시합니다.
#    (load_from_checkpoint가 hparams를 자동으로 불러오지만, 명시적으로 필요할 수 있습니다.)
#    예: LEARNING_RATE, WEIGHT_DECAY 등

os.makedirs(SAVE_DIR, exist_ok=True)


# ===================================================================
# 모델 로드 및 변환
# ===================================================================
# 1. .ckpt 파일로부터 학습된 모델 전체를 불러옵니다.
# BaseTagger 클래스가 현재 스크립트에 정의되어 있어야 합니다.
full_model = BaseTagger.load_from_checkpoint(CKPT_PATH)
full_model.eval() # 평가 모드로 설정

# 2. 모델에서 KcELECTRA 베이스 모델과 토크나이저를 저장합니다.
#    이 과정에서 config.json과 pytorch_model.bin 파일이 생성됩니다.
full_model.electra.save_pretrained(SAVE_DIR)

# 3. 토크나이저 파일도 함께 저장해줍니다.
tokenizer = AutoTokenizer.from_pretrained("beomi/KcELECTRA-base")
tokenizer.save_pretrained(SAVE_DIR)

# 4. 가장 중요한 부분: 커스텀 분류기(classifier)의 가중치(state_dict)를 별도로 저장합니다.
classifier_path = os.path.join(SAVE_DIR, "classifier_state.bin")
torch.save(full_model.classifier.state_dict(), classifier_path)

print(f"모델 변환 완료! 파일이 '{SAVE_DIR}' 디렉토리에 저장되었습니다.")
print("저장된 파일 목록:")
for filename in os.listdir(SAVE_DIR):
    print(f"- {filename}")

tokenizer_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

모델 변환 완료! 파일이 './hf_kpoem' 디렉토리에 저장되었습니다.
저장된 파일 목록:
- config.json
- model.safetensors
- tokenizer_config.json
- special_tokens_map.json
- tokenizer.json
- classifier_state.bin


In [7]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import os

# ===================================================================
# 사용자 노트북의 기존 클래스 및 변수 (실행 전 확인 필요)
# ===================================================================
# BaseTagger 클래스 정의가 필요합니다.
# class BaseTagger(pl.LightningModule):
#     ... 

# LABELS 리스트 정의가 필요합니다.
LABELS = ['불평/불만', '환영/호의', '감동/감탄', '지긋지긋', '고마움', '슬픔', '화남/분노', '존경', '기대감', '우쭐댐/무시함', '안타까움/실망', '비장함', '의심/불신', '뿌듯함', '편안/쾌적', '신기함/관심', '아껴주는', '부끄러움', '공포/무서움', '절망', '한심함', '역겨움/징그러움', '짜증', '어이없음', '없음', '패배/자기혐오', '귀찮음', '힘듦/지침', '즐거움/신남', '깨달음', '죄책감', '증오/혐오', '흐뭇함(귀여움/예쁨)', '당황/난처', '경악', '부담/안_내킴', '서러움', '재미없음', '불쌍함/연민', '놀람', '행복', '불안/걱정', '기쁨', '안심/신뢰']
NUM_LABELS = len(LABELS)


# ===================================================================
# Hugging Face 모델 로드를 위한 재사용 클래스 정의 (forward 함수 수정)
# ===================================================================
class CustomKcELECTRAForSequenceClassification(nn.Module):
    def __init__(self, model_path, num_labels, classifier_dropout_rate=0.1):
        super().__init__()
        self.electra = AutoModel.from_pretrained(model_path)
        self.classifier = nn.Sequential(
            nn.Dropout(p=classifier_dropout_rate),
            nn.Linear(self.electra.config.hidden_size, num_labels)
        )

    # ==================== 주요 수정 부분 ====================
    # forward 함수가 token_type_ids도 받을 수 있도록 수정
    def forward(self, input_ids, attention_mask, token_type_ids=None, **kwargs):
        outputs = self.electra(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids # 받은 token_type_ids를 electra 모델에 전달
        )
        # ========================================================
        pooled_output = outputs.last_hidden_state[:, 0]
        logits = self.classifier(pooled_output)
        return logits

# ===================================================================
# 비교 검증 시작
# ===================================================================
# 0. 사용할 장치 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용할 장치: {device}")

# 1. 경로 설정
CKPT_PATH = "./model/model_B/best_model_B_minmax_0.2.ckpt"
HF_PATH = "./hf_kpoem"

# 2. 원본 모델(.ckpt) 로드 및 장치로 이동
print("\n1. 원본 .ckpt 모델 로드 중...")
try:
    original_model = BaseTagger.load_from_checkpoint(CKPT_PATH, map_location=device)
    original_model.to(device)
    original_model.eval()
    print(" -> 원본 모델 로드 및 장치 이동 완료")
except NameError:
    print("\n[오류] 'BaseTagger' 클래스가 정의되지 않았습니다.")
    exit()

# 3. 변환된 Hugging Face 모델 로드 및 장치로 이동
print("\n2. 변환된 Hugging Face 모델 로드 중...")
converted_model = CustomKcELECTRAForSequenceClassification(HF_PATH, NUM_LABELS)
classifier_weights_path = os.path.join(HF_PATH, "classifier_state.bin")
converted_model.classifier.load_state_dict(torch.load(classifier_weights_path, map_location=device))
converted_model.to(device)
converted_model.eval()
print(" -> 변환된 모델 로드 및 장치 이동 완료")

# 4. 동일한 입력 데이터 준비
print("\n3. 비교를 위한 입력 데이터 준비 중...")
tokenizer = AutoTokenizer.from_pretrained(HF_PATH)
sample_text = "이런 결과를 보니 정말 뿌듯하고, 앞으로가 더 기대되네요."
inputs = tokenizer(sample_text, return_tensors="pt", max_length=512, truncation=True, padding=True)
print(f" -> 샘플 텍스트: \"{sample_text}\"")

# 5. 입력 텐서를 모델과 동일한 장치로 이동
inputs = {k: v.to(device) for k, v in inputs.items()}

# 6. 각 모델로 추론 실행
print("\n4. 각 모델에서 추론 실행 및 결과 비교...")
with torch.no_grad():
    # 원본 모델 추론
    original_output = original_model.electra(
        inputs['input_ids'], 
        attention_mask=inputs['attention_mask'],
        token_type_ids=inputs['token_type_ids'] # 원본 모델에도 token_type_ids 전달
    )
    original_logits = original_model.classifier(original_output.last_hidden_state[:, 0, :])

    # 변환된 모델 추론
    converted_logits = converted_model(**inputs)

# 7. 출력 값 비교
are_identical = torch.allclose(original_logits, converted_logits, atol=1e-6)

print("-" * 50)
print("비교 결과:")
print(f" - 원본 모델 출력 (첫 5개 값): {original_logits[0, :5].tolist()}")
print(f" - 변환된 모델 출력 (첫 5개 값): {converted_logits[0, :5].tolist()}")

if are_identical:
    print("\n성공: 두 모델의 출력(logits)이 일치합니다. 변환이 성공적으로 완료되었습니다.")
else:
    print("\n실패: 두 모델의 출력이 다릅니다. 변환 과정을 다시 확인해주세요.")
    difference = torch.abs(original_logits - converted_logits).max()
    print(f"   (최대 오차: {difference.item()})")
print("-" * 50)

사용할 장치: cuda

1. 원본 .ckpt 모델 로드 중...
 -> 원본 모델 로드 및 장치 이동 완료

2. 변환된 Hugging Face 모델 로드 중...
 -> 변환된 모델 로드 및 장치 이동 완료

3. 비교를 위한 입력 데이터 준비 중...
 -> 샘플 텍스트: "이런 결과를 보니 정말 뿌듯하고, 앞으로가 더 기대되네요."

4. 각 모델에서 추론 실행 및 결과 비교...
--------------------------------------------------
비교 결과:
 - 원본 모델 출력 (첫 5개 값): [-3.960331678390503, 1.0175210237503052, 2.4551615715026855, -4.356122016906738, -0.8307849168777466]
 - 변환된 모델 출력 (첫 5개 값): [-3.960331678390503, 1.0175210237503052, 2.4551615715026855, -4.356122016906738, -0.8307849168777466]

✅ 성공: 두 모델의 출력(logits)이 일치합니다. 변환이 성공적으로 완료되었습니다.
--------------------------------------------------
